In [ ]:
# ============================================================
# Lab Assignment 2 - Banker's Algorithm for Deadlock Avoidance
# Course: Operating System Lab ENCA252 | BCA (AI & DS)
# ============================================================

# ─────────────────────────────────────────
# TASK 1: System Input and Data Representation
# ─────────────────────────────────────────

def get_default_data():
    """
    Default system state for demonstration.

    5 Processes (P0–P4), 3 Resource Types (A, B, C)

    Classic Banker's Algorithm example from Silberschatz OS textbook.
    """
    n = 5   # Number of processes
    m = 3   # Number of resource types

    # Allocation Matrix: resources currently held by each process
    allocation = [
        [0, 1, 0],   # P0
        [2, 0, 0],   # P1
        [3, 0, 2],   # P2
        [2, 1, 1],   # P3
        [0, 0, 2],   # P4
    ]

    # Maximum Matrix: maximum resources each process may ever request
    maximum = [
        [7, 5, 3],   # P0
        [3, 2, 2],   # P1
        [9, 0, 2],   # P2
        [2, 2, 2],   # P3
        [4, 3, 3],   # P4
    ]

    # Available resources vector
    available = [3, 3, 2]

    return n, m, allocation, maximum, available


def get_user_data():
    """Accept system data interactively from the user."""
    print("\n" + "="*55)
    print("   TASK 1: System Input")
    print("="*55)

    n = int(input("  Enter number of processes : "))
    m = int(input("  Enter number of resources : "))

    print(f"\n  Enter Allocation Matrix ({n} x {m}):")
    allocation = []
    for i in range(n):
        row = list(map(int, input(f"    P{i} : ").split()))
        allocation.append(row)

    print(f"\n  Enter Maximum Matrix ({n} x {m}):")
    maximum = []
    for i in range(n):
        row = list(map(int, input(f"    P{i} : ").split()))
        maximum.append(row)

    print(f"\n  Enter Available Resources (1 x {m}):")
    available = list(map(int, input("    Available : ").split()))

    return n, m, allocation, maximum, available


def display_system_state(n, m, allocation, maximum, available, need=None):
    """
    Display full system state in a formatted table.
    Shows Allocation, Maximum, Need (if computed), and Available.
    """
    res_labels = [chr(65 + i) for i in range(m)]   # A, B, C, ...

    print("\n" + "="*75)
    print("   System State")
    print("="*75)

    # Header
    alloc_h = "  ".join(res_labels)
    max_h   = "  ".join(res_labels)
    need_h  = "  ".join(res_labels) if need else ""

    header = f"  {'PID':<6} {'Allocation':<{3*m+4}} {'Maximum':<{3*m+4}}"
    if need:
        header += f" {'Need':<{3*m+4}}"
    print(header)
    print(f"  {'':6} {alloc_h:<{3*m+4}} {max_h:<{3*m+4}}", end="")
    if need:
        print(f" {need_h:<{3*m+4}}", end="")
    print()
    print("  " + "-"*65)

    for i in range(n):
        alloc_str = "  ".join(str(x) for x in allocation[i])
        max_str   = "  ".join(str(x) for x in maximum[i])
        row = f"  P{i:<5} {alloc_str:<{3*m+4}} {max_str:<{3*m+4}}"
        if need:
            need_str = "  ".join(str(x) for x in need[i])
            row += f" {need_str:<{3*m+4}}"
        print(row)

    print("  " + "-"*65)
    avail_str = "  ".join(str(x) for x in available)
    print(f"  Available Resources : {avail_str}  ({' / '.join(res_labels)})")
    print("="*75)


# ─────────────────────────────────────────
# TASK 2: Need Matrix Calculation
# ─────────────────────────────────────────

def calculate_need(n, m, allocation, maximum):
    """
    Compute the Need Matrix.

    Formula: Need[i][j] = Maximum[i][j] - Allocation[i][j]

    The Need matrix represents how many more resources of each
    type process i may still request before completing.
    """
    need = []
    for i in range(n):
        row = [maximum[i][j] - allocation[i][j] for j in range(m)]
        need.append(row)
    return need


def display_need_matrix(n, m, need):
    """Display the computed Need Matrix."""
    res_labels = [chr(65 + i) for i in range(m)]
    print("\n" + "="*40)
    print("   TASK 2: Need Matrix  (Max − Allocation)")
    print("="*40)
    print(f"  {'PID':<6} " + "  ".join(res_labels))
    print("  " + "-"*30)
    for i in range(n):
        need_str = "  ".join(str(x) for x in need[i])
        print(f"  P{i:<5} {need_str}")
    print("="*40)


# ─────────────────────────────────────────
# TASK 3 & 4: Safety Algorithm + Safe Sequence
# ─────────────────────────────────────────

def bankers_safety_algorithm(n, m, need, allocation, available):
    """
    Banker's Safety Algorithm.

    Steps:
      1. Work  = copy of Available
      2. Finish[i] = False for all i
      3. Find index i such that:
           - Finish[i] == False
           - Need[i] <= Work  (element-wise)
      4. If found:
           Work   = Work + Allocation[i]   (simulate release)
           Finish[i] = True
           Add i to safe sequence
      5. Repeat step 3 until no such i found.
      6. If all Finish[i] == True → SAFE state.
         Otherwise                → UNSAFE state.

    Returns:
      is_safe      (bool)  : True if system is in a safe state.
      safe_sequence(list)  : Execution order if safe, else partial order.
      trace        (list)  : Step-by-step log for display.
    """
    work   = available[:]          # Step 1: Work = Available
    finish = [False] * n           # Step 2: all unfinished
    safe_sequence = []
    trace = []                     # Audit trail for display

    step = 1
    while len(safe_sequence) < n:
        allocated_this_round = False

        for i in range(n):
            if finish[i]:
                continue

            # Check if Need[i] <= Work (element-wise)
            if all(need[i][j] <= work[j] for j in range(m)):
                # Simulate: grant resources and process completes
                work_before = work[:]
                work = [work[j] + allocation[i][j] for j in range(m)]
                finish[i] = True
                safe_sequence.append(i)
                allocated_this_round = True

                trace.append({
                    "step"      : step,
                    "process"   : i,
                    "need"      : need[i][:],
                    "work_before": work_before,
                    "allocation": allocation[i][:],
                    "work_after": work[:],
                })
                step += 1
                break   # Restart search after each allocation

        if not allocated_this_round:
            break   # No eligible process found → deadlock risk

    is_safe = all(finish)
    return is_safe, safe_sequence, trace


def display_safety_trace(trace, m):
    """Display the step-by-step safety algorithm execution."""
    res_labels = [chr(65 + i) for i in range(m)]
    rl = " ".join(res_labels)

    print("\n" + "="*80)
    print("   TASK 3: Safety Algorithm — Step-by-Step Trace")
    print("="*80)
    print(f"  {'Step':<6} {'Process':<10} {'Need':^{3*m+2}} {'Work Before':^{3*m+4}} "
          f"{'Alloc':^{3*m+2}} {'Work After':^{3*m+4}}")
    print(f"  {'':6} {'':10} {rl:^{3*m+2}} {rl:^{3*m+4}} "
          f"{rl:^{3*m+2}} {rl:^{3*m+4}}")
    print("  " + "-"*75)

    for t in trace:
        need_s   = " ".join(str(x) for x in t["need"])
        wb_s     = " ".join(str(x) for x in t["work_before"])
        alloc_s  = " ".join(str(x) for x in t["allocation"])
        wa_s     = " ".join(str(x) for x in t["work_after"])
        print(f"  {t['step']:<6} P{t['process']:<9} {need_s:^{3*m+2}} "
              f"{wb_s:^{3*m+4}} {alloc_s:^{3*m+2}} {wa_s:^{3*m+4}}")

    print("="*80)


def display_safe_sequence(is_safe, safe_sequence):
    """Display the safe sequence result (Task 4)."""
    print("\n" + "="*55)
    print("   TASK 4: Safe Sequence Determination")
    print("="*55)

    if is_safe:
        seq = " → ".join(f"P{i}" for i in safe_sequence)
        print(f"\n  ✅ System is in a SAFE STATE.\n")
        print(f"  Safe Sequence : {seq}\n")
        print("  Interpretation:")
        print("  Each process in this order can obtain all the")
        print("  resources it needs, execute, and release them")
        print("  for the next process — no deadlock will occur.")
    else:
        partial = " → ".join(f"P{i}" for i in safe_sequence)
        print(f"\n  ❌ System is in an UNSAFE STATE.\n")
        if partial:
            print(f"  Partial sequence before deadlock : {partial}")
        print("  No safe sequence exists. Deadlock may occur.")

    print("="*55)


# ─────────────────────────────────────────
# TASK 5: Result Analysis
# ─────────────────────────────────────────

def result_analysis(is_safe, safe_sequence, n, m, need, allocation, available):
    """
    Analyze and explain the result of Banker's Algorithm.
    Compare with theoretical expectations.
    """
    print("\n" + "="*65)
    print("   TASK 5: Result Analysis")
    print("="*65)

    total_alloc   = [sum(allocation[i][j] for i in range(n)) for j in range(m)]
    total_max     = [sum(need[i][j] + allocation[i][j] for i in range(n)) for j in range(m)]
    resource_util = [f"{total_alloc[j]}/{total_max[j]}" for j in range(m)]

    res_labels = [chr(65 + i) for i in range(m)]
    print(f"\n  Resource Utilization : " +
          "  ".join(f"{res_labels[j]}={resource_util[j]}" for j in range(m)))
    print(f"  Available            : " +
          "  ".join(f"{res_labels[j]}={available[j]}" for j in range(m)))

    print("\n  ─── Theoretical Comparison ───────────────────────────")
    print("  Banker's Algorithm guarantees deadlock avoidance by")
    print("  never granting a request that would leave the system")
    print("  in an unsafe state — even if resources are available.")
    print()

    if is_safe:
        print("  Result  : SAFE STATE ✅")
        print("  Reason  : A valid safe sequence was found.")
        print("  Theory  : In a safe state, there is always at least")
        print("            one execution order where all processes can")
        print("            finish. The system can satisfy every future")
        print("            request without risking deadlock.")
        print()
        print("  Key Insight:")
        print("    The safe sequence does NOT mean processes must run")
        print("    in that exact order — it proves that such an order")
        print("    EXISTS, guaranteeing no deadlock is possible.")
    else:
        print("  Result  : UNSAFE STATE ❌")
        print("  Reason  : No valid safe sequence could be found.")
        print("  Theory  : An unsafe state means the system MIGHT")
        print("            deadlock — not that it definitely will.")
        print("            The OS should deny the resource request")
        print("            that caused this state transition.")
        print()
        print("  Key Insight:")
        print("    Unsafe ≠ Deadlock (necessarily), but Banker's")
        print("    Algorithm conservatively rejects such states to")
        print("    guarantee deadlock never occurs.")

    print()
    print("  Limitations of Banker's Algorithm:")
    print("    • Requires knowing maximum resource needs in advance.")
    print("    • Processes must not exceed declared maximum.")
    print("    • Number of processes and resources must be fixed.")
    print("    • Overhead makes it impractical for real-time OS.")
    print("="*65)


# ─────────────────────────────────────────
# MAIN — Entry Point
# ─────────────────────────────────────────

def main():
    print("\n" + "★"*60)
    print("  OS Lab Assignment 2 — Banker's Algorithm")
    print("  Deadlock Avoidance  |  ENCA252  |  BCA (AI & DS)")
    print("★"*60)

    print("\n  [1] Use default sample data (Silberschatz example)")
    print("  [2] Enter your own data")
    choice = input("\n  Your choice (1/2): ").strip()

    if choice == "2":
        n, m, allocation, maximum, available = get_user_data()
    else:
        n, m, allocation, maximum, available = get_default_data()
        print("\n  Using default sample data.")

    # ── Task 2: Need Matrix ──
    need = calculate_need(n, m, allocation, maximum)
    display_need_matrix(n, m, need)

    # ── Task 1: Full System State (shown after need is computed) ──
    display_system_state(n, m, allocation, maximum, available, need)

    # ── Tasks 3 & 4: Safety Algorithm + Safe Sequence ──
    is_safe, safe_sequence, trace = bankers_safety_algorithm(
        n, m, need, allocation, available
    )
    display_safety_trace(trace, m)
    display_safe_sequence(is_safe, safe_sequence)

    # ── Task 5: Analysis ──
    result_analysis(is_safe, safe_sequence, n, m, need, allocation, available)

    print("\n  ✅ Assignment Complete!\n")


if __name__ == "__main__":
    main()